### Generating the explanations that humans and LLM judges will later score.

For every selected recipe-profile pair from Step 4 (2 per profile, 120 cases) this notebook writes **four** explanations:

| Technique | What the writer gets | What it does |
|---|---|---|
| `template` | verified facts only (no LLM) | fixed jinja2 sentences |
| `direct_llm` | profile card + recipe card | the LLM writes freely |
| `grounded_llm` | a numbered permitted-fact list (F1, F2, ...) | the LLM may use only those facts and cites them |
| `grounded_check_llm` | the grounded draft | each claim is audited against the facts, then revised (max 2 rounds) |

120 cases x 4 techniques = **480 explanations**. Optionally ~20 goal-framed grounded explanations are generated as extras (not part of the 480).

#### Usage
1. **Prerequisites** - Steps 2-4 finished: `processed/recommendation_pool.jsonl`, `profiles_historical.jsonl`, `profiles_synthetic.jsonl`, `catalog.parquet` exist, plus `processed/constraint_check_sheet.csv`. I have filled out the last column of `constraint_check_sheet.csv`. A more compact .CSV, i.e., `constraint_overrides.csv` in generated and used in this notebook. The notebook lives in the project root next to `src/`; `src/config.py` must be present.
2. **Ollama running** - install Ollama, run `ollama pull llama3.1:8b` once, and make sure the Ollama app/server is running.
3. Run the cells **top to bottom**:
   * Section 1 (setup) and Sections 2-4 (code) only *define* things: run them once after opening the notebook (or after a kernel restart). They do nothing visible.
   * Section 5: edit the settings.
   * Section 6: **test the connection** (must pass before anything else).
   * Section 7: **dry run** (no model calls, ~seconds) -> check the prompts.
   * Section 8: **pilot** (6 cases) -> read the explanations.
   * Section 9: **full run** (resumable: stop the cell any time, run it again and it continues).
   * Section 10: **inspect the results**.
4. Existing results are **kept and resumed**: if `processed/explanations.jsonl` already exists, the pilot and full run continue from it. If you change the profiles or recommendations (Steps 3-4), run once with `fresh=True` (the cache keeps unchanged prompts free). Back up `explanations.jsonl` first.
5. The pilot and the dry run rewrite `processed/case_inputs.jsonl` with only their subset of cases; **finish with the full-run cell** (Section 9) so `case_inputs.jsonl` covers all 120 cases before Step 6.

> Inputs : 
* processed/recommendation_pool.jsonl
* processed/profiles_historical.jsonl 
* processed/profiles_synthetic.jsonl
* processed/catalog.parquet
* processed/constraint_check_sheet.csv

> Outputs: 
* processed/case_inputs.jsonl
* processed/explanations.jsonl
* processed/explanations_summary.csv
* processed/generation_report.json
* processed/prompt_preview.txt (dry run)
* processed/constraint_overrides.csv

#### 1. Setup: paths and imports

In [1]:
import sys, os, re, json, time, random, hashlib, itertools
from collections import defaultdict, deque, Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from jinja2 import Environment, StrictUndefined


def find_project_root() -> Path:
    """The folder that contains src/config.py (this notebook may sit in the project root or in a sub-folder)."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src" / "config.py").exists():
            return p
    raise FileNotFoundError(f"Could not find src/config.py above {here}. Open the notebook from inside the FoodJudgeBench folder.")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
from config import PROC_DIR, GOAL_RULES, as_list, read_table, scrub_text    # shared helpers from Step 2-4

CACHE_DIR = ROOT / "processed" / "llm_cache"
print("project root:", ROOT)
print("processed dir:", PROC_DIR)


project root: C:\HP Data\University-Masters\Semester_3\P1_Natural Language Processing\Lab\Project\FoodJudgeBench
processed dir: C:\HP Data\University-Masters\Semester_3\P1_Natural Language Processing\Lab\Project\FoodJudgeBench\processed


#### LLM Clients
One client for every provider: rate limiting, retries, a disk cache (`processed/llm_cache/`, re-runs are free), and a clean `QuotaExhausted` stop.
Providers include `ollama` (local, no key, no quota), the free cloud APIs, and `mock` (offline fake for testing). Definitions only - nothing runs here.

In [2]:
import dotenv

dotenv.load_dotenv()

True

In [3]:
# base URLs: Hugging Face's is from its docs; the others are the providers' documented OpenAI-compatible endpoints
# (not testable from the build sandbox - use --ping). default_model=None means "you must pass --model".
PROVIDERS = {
    "groq": {"base_url": "https://api.groq.com/openai/v1", "key_env": "GROQ_API_KEY", "default_model": "openai/gpt-oss-120b",
             "rpm": 25, "tpm": 7000},          # free plan (docs): 30 RPM, 8K TPM, 1K RPD, 200K TPD for gpt-oss-120b
    "gemini": {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai", "key_env": "GEMINI_API_KEY",
               "default_model": "gemini-2.5-flash", "rpm": 8, "tpm": None},   # free limits are shown in AI Studio, not published
    "openrouter": {"base_url": "https://openrouter.ai/api/v1", "key_env": "OPENROUTER_API_KEY", "default_model": None,
                   "rpm": 18, "tpm": None},    # ':free' models: 20 RPM and 50 requests/day (1000/day after a $10 top-up)
    "nvidia": {"base_url": "https://integrate.api.nvidia.com/v1", "key_env": "NVIDIA_API_KEY", "default_model": None,
               "rpm": 35, "tpm": None},        # NIM free tier reported ~40 RPM; needs NVIDIA developer membership
    "cerebras": {"base_url": "https://api.cerebras.ai/v1", "key_env": "CEREBRAS_API_KEY", "default_model": None, "rpm": 25, "tpm": None},
    "mistral": {"base_url": "https://api.mistral.ai/v1", "key_env": "MISTRAL_API_KEY", "default_model": None, "rpm": 20, "tpm": None},
    "huggingface": {"base_url": "https://router.huggingface.co/v1", "key_env": "HF_TOKEN", "default_model": "openai/gpt-oss-120b",
                    "rpm": 20, "tpm": None},   # NOT free: needs purchased credits or PRO (see docs); model suffix picks provider
    # local Ollama (OpenAI-compatible endpoint, no key, no quota): set OLLAMA_HOST to use another machine, e.g. http://192.168.1.5:11434
    "ollama": {"base_url": os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/").removesuffix("/v1") + "/v1", "key_env": None,
               "default_model": None, "rpm": 600, "tpm": None},
    "custom": {"base_url": os.environ.get("FJB_BASE_URL", ""), "key_env": "FJB_API_KEY", "default_model": None, "rpm": 20, "tpm": None},
}


In [4]:
class QuotaExhausted(RuntimeError):
    """Daily/monthly quota hit (or a wait longer than max_wait was requested). Progress is saved; re-run later."""


@dataclass
class LLMResult:
    text: str
    model: str
    usage: dict = field(default_factory=dict)
    cached: bool = False
    latency: float = 0.0


def load_env_file(path: Path | None = None) -> None:
    """Minimal .env loader (KEY=VALUE per line); real environment variables win."""
    p = path or ROOT / ".env"
    if not p.exists():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))


def strip_thinking(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S | re.I)
    return re.sub(r"^.*?</think>", "", text, flags=re.S | re.I).strip() if "</think>" in text else text.strip()


class _Throttle:
    """Sliding-window limiter for requests/minute and tokens/minute."""

    def __init__(self, rpm, tpm):
        self.rpm, self.tpm = rpm, tpm
        self.events: deque = deque()          # (timestamp, tokens)

    def wait(self, est_tokens: int) -> None:
        while True:
            now = time.time()
            while self.events and now - self.events[0][0] >= 60:
                self.events.popleft()
            n_req = len(self.events)
            n_tok = sum(t for _, t in self.events)
            need = 0.0
            if self.rpm and n_req >= self.rpm:
                need = max(need, 60 - (now - self.events[0][0]))
            if self.tpm and self.events and n_tok + est_tokens > self.tpm:
                need = max(need, 60 - (now - self.events[0][0]))
            if need <= 0:
                return
            time.sleep(min(need + 0.05, 30))

    def record(self, tokens: int) -> None:
        self.events.append((time.time(), tokens))


In [5]:
class LLMClient:
    def __init__(self, provider: str, model: str | None = None, rpm: int | None = None, tpm: int | None = None,
                 extra: dict | None = None, max_wait: float = 120.0, cache: bool = True, timeout: float = 180.0):
        if provider not in PROVIDERS:
            raise ValueError(f"unknown provider '{provider}'. choose from {sorted(PROVIDERS) + ['mock']}")
        cfg = PROVIDERS[provider]
        self.provider = provider
        self.model = model or os.environ.get("FJB_MODEL") or cfg["default_model"]
        if not self.model:
            raise ValueError(f"provider '{provider}' has no default model: pass --model (see the provider's model list)")
        self.base_url = cfg["base_url"].rstrip("/")
        if not self.base_url:
            raise ValueError("custom provider needs FJB_BASE_URL")
        self.key = os.environ.get(cfg["key_env"], "") if cfg["key_env"] else "ollama"     # Ollama ignores the key
        if provider == "ollama":
            timeout = max(timeout, 900.0)                          # local models can take minutes per call on CPU
        if not self.key:
            raise RuntimeError(f"missing API key: set {cfg['key_env']} in the environment or in {ROOT / '.env'}")
        self.extra = extra or {}
        self.max_wait, self.use_cache, self.timeout = max_wait, cache, timeout
        self.throttle = _Throttle(rpm if rpm is not None else cfg["rpm"], tpm if tpm is not None else cfg["tpm"])
        self.calls = self.cache_hits = 0
        self.tokens_in = self.tokens_out = 0

    # ------------------------------------------------------------------
    def _key(self, body: dict) -> str:
        blob = json.dumps({"p": self.provider, "b": {k: v for k, v in body.items()}}, sort_keys=True, ensure_ascii=False)
        return hashlib.sha256(blob.encode()).hexdigest()

    def chat(self, messages, task: str = "generic", temperature: float = 0.4, max_tokens: int = 900, meta=None) -> LLMResult:
        """`task` and `meta` are only used by the offline mock; real providers ignore them."""
        import requests

        body = {"model": self.model, "messages": messages, "temperature": temperature, "max_tokens": max_tokens, **self.extra}
        ck = self._key(body)
        cpath = CACHE_DIR / self.provider / f"{ck}.json"
        if self.use_cache and cpath.exists():
            d = json.loads(cpath.read_text(encoding="utf-8"))
            self.cache_hits += 1
            return LLMResult(d["text"], d["model"], d.get("usage", {}), True, 0.0)

        est = sum(len(m["content"]) for m in messages) // 3 + min(max_tokens, 500)
        url = f"{self.base_url}/chat/completions"
        headers = {"Authorization": f"Bearer {self.key}", "Content-Type": "application/json"}
        tweaked: set[str] = set()
        for attempt in range(7):
            self.throttle.wait(est)
            t0 = time.time()
            try:
                r = requests.post(url, headers=headers, json=body, timeout=self.timeout)
            except requests.exceptions.RequestException as e:
                self._sleep_or_raise(min(60, 2 ** attempt + random.random()), f"network error: {e}", attempt, 5)
                continue
            self.throttle.record(est)
            if r.status_code == 200:
                data = r.json()
                try:
                    text = strip_thinking(data["choices"][0]["message"]["content"] or "")
                except (KeyError, IndexError, TypeError):
                    raise RuntimeError(f"unexpected response shape: {str(data)[:300]}")
                usage = data.get("usage") or {}
                self.calls += 1
                self.tokens_in += usage.get("prompt_tokens", 0)
                self.tokens_out += usage.get("completion_tokens", 0)
                if usage.get("total_tokens"):                      # replace the estimate with the real figure
                    self.throttle.events[-1] = (self.throttle.events[-1][0], usage["total_tokens"])
                if not text:
                    self._sleep_or_raise(2 + attempt, "empty completion (reasoning budget used up? raise --max-tokens)", attempt, 3)
                    continue
                res = LLMResult(text, data.get("model", self.model), usage, False, time.time() - t0)
                if self.use_cache:
                    cpath.parent.mkdir(parents=True, exist_ok=True)
                    cpath.write_text(json.dumps({"text": text, "model": res.model, "usage": usage}, ensure_ascii=False), encoding="utf-8")
                return res
            if r.status_code == 429:
                wait = _retry_after(r) or min(60, 2 ** attempt * 2 + random.random())
                low = r.text.lower()
                if wait > self.max_wait or any(w in low for w in ("per day", "daily", "tpd", "rpd", "quota exceeded")) and wait > 30:
                    raise QuotaExhausted(f"{self.provider}: rate limit says wait {wait:.0f}s ({r.text[:200]})")
                self._sleep_or_raise(wait, f"429 rate limited ({r.text[:120]})", attempt, 6)
                continue
            if r.status_code in (401, 403):
                raise RuntimeError(f"{self.provider}: authentication failed ({r.status_code}): {r.text[:200]}")
            if r.status_code == 400:                                # unsupported parameter: adapt once each, else give up
                low = r.text.lower()
                if "max_tokens" in low and "max_completion_tokens" in low and "max_tokens" in body and "mct" not in tweaked:
                    body["max_completion_tokens"] = body.pop("max_tokens"); tweaked.add("mct"); continue
                if "temperature" in low and "temperature" in body and "temp" not in tweaked:
                    body.pop("temperature"); tweaked.add("temp"); continue
                raise RuntimeError(f"{self.provider}: bad request: {r.text[:300]}")
            if r.status_code == 404:
                raise RuntimeError(f"{self.provider}: not found (404) - check the model name '{self.model}': {r.text[:200]}")
            self._sleep_or_raise(min(60, 2 ** attempt + random.random()), f"HTTP {r.status_code}: {r.text[:120]}", attempt, 5)
        raise RuntimeError(f"{self.provider}: gave up after repeated failures")

    def _sleep_or_raise(self, wait: float, why: str, attempt: int, max_attempts: int) -> None:
        if attempt + 1 >= max_attempts:
            raise RuntimeError(f"{self.provider}: {why} (after {attempt + 1} attempts)")
        print(f"    [{self.provider}] {why}; waiting {wait:.0f}s", file=sys.stderr)
        time.sleep(wait)

    def stats(self) -> dict:
        return {"provider": self.provider, "model": self.model, "api_calls": self.calls, "cache_hits": self.cache_hits,
                "prompt_tokens": self.tokens_in, "completion_tokens": self.tokens_out}


def _retry_after(r) -> float | None:
    v = r.headers.get("retry-after") or r.headers.get("Retry-After")
    try:
        return float(v) if v is not None else None
    except ValueError:
        return None


In [6]:
# offline mock (for dry runs and tests)
class MockClient:
    """Deterministic stand-in: exercises the whole pipeline (including the check/revise loop) without any API.
    It deliberately writes one invented sentence into about half of the grounded drafts and omits a must-mention fact
    in about a third, so the self-check technique has something to find."""

    provider, model = "mock", "mock-1"

    def __init__(self, *a, **k):
        self.calls = self.cache_hits = 0
        self.tokens_in = self.tokens_out = 0

    def stats(self) -> dict:
        return {"provider": "mock", "model": "mock-1", "api_calls": self.calls, "cache_hits": 0, "prompt_tokens": 0, "completion_tokens": 0}

    def chat(self, messages, task="generic", temperature=0.4, max_tokens=900, meta=None) -> LLMResult:
        self.calls += 1
        meta = meta or {}
        h = int(hashlib.md5(json.dumps(meta.get("title", "") + str(meta.get("case", "")), default=str).encode()).hexdigest(), 16)
        facts = meta.get("facts", [])
        if task == "explain_direct":
            ings = ", ".join(meta.get("ingredients", [])[:3])
            text = (f"{meta['title']} looks like a good match for you. It uses {ings} and takes about {meta.get('minutes', 30)} minutes, "
                    f"so it should fit a busy evening. Many people enjoy it as a relaxed family dinner, and it is packed with flavour.")
        elif task == "explain_grounded":
            by = {f["id"]: f for f in facts}
            recipe = next(f for f in facts if f["kind"] == "recipe")
            parts = [f"The recipe is {meta['title']} [{recipe['id']}]."]
            musts = [f for f in facts if f["must_mention"]]
            if h % 3 == 0 and len(musts) > 0:
                musts = musts[1:]                                           # omit one must-mention fact
            for f in facts:
                if f["kind"] == "match" and not f["must_mention"]:
                    parts.append(f"{f['text'].rstrip('.')} [{f['id']}].")
            for f in musts:
                parts.append(f"A trade-off to know: {f['text'].rstrip('.')} [{f['id']}].")
            if h % 2 == 0:
                parts.append(f"It is also great for family dinners [{recipe['id']}].")
            text = " ".join(parts[:7])
        elif task == "check":
            sents = re.split(r"(?<=[.!?])\s+", meta.get("text", ""))
            claims = []
            for s in sents:
                bad = "family" in s.lower()
                claims.append({"claim": s, "verdict": "unsupported" if bad else "supported", "fact_ids": [] if bad else ["F1"]})
            text = "```json\n" + json.dumps({"claims": claims}) + "\n```"
        elif task == "revise":
            sents = re.split(r"(?<=[.!?])\s+", meta.get("text", ""))
            keep = [s for s in sents if "family" not in s.lower()]
            by = {f["id"]: f for f in facts}
            for fid in meta.get("missing_ids", []):
                keep.append(f"A trade-off to know: {by[fid]['text'].rstrip('.')} [{fid}].")
            text = " ".join(keep)
        elif task == "shorten":
            text = " ".join(meta.get("text", "").split()[:90])
        else:
            text = "ok"
        self.tokens_in += sum(len(m["content"]) for m in messages) // 4
        self.tokens_out += len(text) // 4
        return LLMResult(text, "mock-1", {}, False, 0.0)


In [7]:
def get_client(provider: str, model: str | None = None, **kw):
    load_env_file()
    if provider == "mock":
        return MockClient()
    return LLMClient(provider, model, **kw)


def ollama_check(model: str) -> bool:
    """Is the Ollama server reachable, and is `model` pulled?"""
    import requests
    base = PROVIDERS["ollama"]["base_url"].rstrip("/").removesuffix("/v1")
    try:
        r = requests.get(base + "/api/tags", timeout=5)
        r.raise_for_status()
    except Exception as e:
        print(f"Cannot reach Ollama at {base}: {type(e).__name__}.\n"
              "  -> start it (Windows/Mac: open the Ollama app; Linux: `ollama serve`), then run this cell again.")
        return False
    names = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama is running at {base}. Installed models: {names or 'none'}")
    ok = model in names or f"{model}:latest" in names
    if not ok:
        print(f"Model '{model}' is NOT installed -> run in a terminal:  ollama pull {model}")
    return ok


def ping(provider: str, model: str | None = None, extra: dict | None = None) -> None:
    """One tiny request to prove the provider, key and model work."""
    c = get_client(provider, model, extra=extra, cache=False)
    t0 = time.time()
    r = c.chat([{"role": "user", "content": "Reply with exactly one word: ok"}], max_tokens=300, temperature=0)
    print(f"{c.provider} / {r.model}: reply={r.text!r} in {time.time() - t0:.1f}s usage={r.usage}")


#### Facts, template technique and text checks
Builds the numbered permitted-fact list (F1...Fn) for each case, renders the **template** technique, cleans ingredient text and runs the cheap deterministic checks (citations, numbers not in the facts). No LLM is called here.

In [8]:
# goal -> (nutrient label, key in nutrition_per_serving, unit)
GOAL_NUTRIENT = {
    "lower sodium": ("sodium", "sodium_mg", "mg"),
    "fewer calories": ("calories", "calories", "kcal"),
    "less sugar": ("sugar", "sugars_g", "g"),
    "less saturated fat": ("saturated fat", "saturated_fat_g", "g"),
    "higher protein": ("protein", "protein_g", "g"),
    "more fiber": ("fiber", "fiber_g", "g"),
}
assert set(GOAL_NUTRIENT) == set(GOAL_RULES), "GOAL_NUTRIENT must cover every goal in common.GOAL_RULES"

_LEVELS = [(0.2, "very low"), (0.4, "low"), (0.6, "moderate"), (0.8, "high"), (1.01, "very high")]


def level(p: float) -> str:
    """Percentile (0-1, share of dataset recipes with LESS of the nutrient) -> plain word."""
    for cut, word in _LEVELS:
        if p < cut:
            return word
    return "very high"


def join_and(items) -> str:
    items = [str(i) for i in items]
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]


In [9]:
# ----------------------------------------------------------------------------- ingredient cleaning
# HUMMUS ingredient strings contain placeholders such as "$template2$" where an alternative ingredient was removed,
# and a few lines that are really cook's notes. Neither may reach an annotator or an LLM.
_TEMPLATE = re.compile(r"\$template\d*\$")
_ASIDE = re.compile(r"\([^)]*\b(?:i|we)\s+(?:like|use|used|prefer)\b[^)]*\)", re.I)
_NOTE_LINE = re.compile(r"\bskip it\b|\bi like to\b", re.I)


def clean_ingredient_line(s) -> str | None:
    s = str(s or "").strip()
    if not s or _NOTE_LINE.search(s):
        return None
    m = _TEMPLATE.search(s)
    if m:
        head = s[: m.start()]
        if not re.search(r"[A-Za-z]{3,}", head):          # the line starts with the placeholder: nothing usable
            return None
        cut = head.rfind(" or ")                           # drop the unknown alternative: "butter or 6 tablespoons $t$"
        s = head[:cut] if cut != -1 else head
    s = _ASIDE.sub("", s)
    s = re.sub(r"\s{2,}", " ", s).strip(" ,;(")
    return s or None


def main_ingredients(recipe: dict, k: int = 12) -> list[str]:
    seen, out = set(), []
    for raw in recipe.get("ingredient_names") or []:
        line = clean_ingredient_line(raw)
        if not line:
            continue
        name = re.sub(r"\([^)]*\)", "", line).split(",")[0].strip().lower()
        if len(name) >= 3 and name not in seen:
            seen.add(name)
            out.append(name)
        if len(out) >= k:
            break
    return out


def ingredient_lines(recipe: dict, k: int = 25) -> list[str]:
    lines = [clean_ingredient_line(x) for x in (recipe.get("ingredients_display") or recipe.get("ingredient_names") or [])]
    return [scrub_text(x) for x in lines if x][:k]


In [10]:
# ----------------------------------------------------------------------------- citations, text clean-up, numeric check
CITE_RE = re.compile(r"\s*\[\s*F\d+(?:\s*[,;/]\s*F\d+)*\s*\]")
_NUM_RE = re.compile(r"(?<![\w.])\d+(?:\.\d+)?")


def strip_citations(t: str) -> str:
    return CITE_RE.sub("", t)


def cited_ids(t: str) -> set[str]:
    out = set()
    for m in CITE_RE.finditer(t):
        out.update(re.findall(r"F\d+", m.group(0)))
    return out


def clean_text(raw: str) -> str:
    """What annotators and judges see: no citations, no markdown, one paragraph."""
    t = strip_citations(str(raw))
    t = re.sub(r"\[\s*MUST\s+MENTION\s*\]", "", t, flags=re.I)
    t = re.sub(r"^\s*(?:explanation|recommendation|answer)\s*:\s*", "", t, flags=re.I)
    t = re.sub(r"[*#`>]+", "", t)
    t = re.sub(r"^\s*[-•]\s*", "", t, flags=re.M)
    t = re.sub(r"\s*\n+\s*", " ", t)
    t = re.sub(r"^\s*You,\s*", "", t, flags=re.I)
    t = t.strip()
    if t:
        t = t[0].upper() + t[1:]
    t = re.sub(r"\s+([.,;:!?])", r"\1", t)
    return re.sub(r"\s{2,}", " ", t).strip()


def word_count(t: str) -> int:
    return len(re.findall(r"\S+", t))


def numeric_mismatches(text: str, facts: list[dict]) -> list[str]:
    """Numbers in the text that do not appear (within rounding) in any fact."""
    allowed = [float(x) for f in facts for x in _NUM_RE.findall(f["text"])]
    bad = set()
    for m in _NUM_RE.findall(strip_citations(text)):
        v = float(m)
        if not any(abs(v - a) <= max(1.0, 0.03 * abs(a)) for a in allowed):
            bad.add(m)
    return sorted(bad, key=float)


def split_sentences(t: str) -> list[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s.strip()]

In [11]:
# ----------------------------------------------------------------------------- facts
def _tendency_phrase(metric: str, direction: str) -> str:
    if metric == "cooking time":
        return "longer cooking times" if direction == "higher" else "shorter cooking times"
    if metric == "overall healthiness score":
        return f"a {direction} overall healthiness score"
    return f"{direction} {metric}"


def build_facts(ci: dict) -> list[dict]:
    """Numbered permitted facts for one case. `ci` is the case-input dict built in 05 (profile fields + pool facts + recipe)."""
    f, n, src = ci["pool_facts"], ci["pool_facts"]["nutrition_per_serving"], ci["profile_source"]
    prof = ci["profile_view"]
    facts: list[dict] = []

    def add(kind, text, must=False):
        facts.append({"id": f"F{len(facts) + 1}", "kind": kind, "text": text, "must_mention": bool(must)})

    # ---- what we know about the person
    if src == "synthetic":
        add("profile", "Every preference listed for this person was stated explicitly by them.")
    else:
        add("profile", f"This person's profile is inferred from {prof['n_reviews']} past recipe reviews; nothing in it was confirmed by the person.")
    for c in prof["constraints"]:
        if c["status"] == "stated":
            add("profile", f"Stated dietary requirement: {c['constraint']}.")
        elif c["status"] == "observed_pattern":
            add("profile", f"Past behaviour suggests, without confirmation, a pattern of choosing {c['constraint']} recipes.")
    g = prof["goal"]
    if g["value"] in GOAL_RULES and g["status"] == "stated":
        add("profile", f"Stated health goal: {g['value']}.")
    elif g["value"] == "none":
        add("profile", "The person stated that they have no particular health goal.")
    elif g["status"] == "observed_pattern":
        add("profile", f"Past reviews hint, with low confidence and no confirmation: {g['value']}.")
    else:
        add("profile", "No health goal is known for this person.")
    t = prof["time"]
    if t["status"] == "stated" and t["value"]:
        add("profile", f"Stated time limit: about {int(t['value'])} minutes or less.")
    elif t["status"] == "stated":
        add("profile", "The person stated that they have no time limit.")
    elif t.get("typical_minutes"):
        add("profile", f"No time limit is known; recipes this person liked typically take about {t['typical_minutes']:.0f} minutes.")
    else:
        add("profile", "No time limit is known for this person.")
    if prof["likes"]:
        names = ", ".join(prof["likes"])
        add("profile", f"Stated food likes: {names}." if src == "synthetic"
            else f"Past reviews suggest the person tends to like recipes featuring: {names} (inferred pattern).")
    else:
        add("profile", "No food likes are known for this person." if src == "synthetic"
            else "No clear taste pattern could be inferred from past reviews.")
    if prof["dislikes"]:
        names = ", ".join(prof["dislikes"])
        add("profile", f"Stated food dislikes: {names}." if src == "synthetic"
            else f"Past reviews suggest the person tends to rate lower recipes featuring: {names} (inferred pattern).")
    for tr in ci["tendencies_sorted"][:3]:
        add("profile", f"Recipes this person liked tend to have {_tendency_phrase(tr['metric'], tr['direction'])} than typical recipes "
                       f"({tr['confidence']} confidence, inferred).")
    if prof.get("tension"):
        add("profile", f"About this person's priorities: {prof['tension']}.")
    unknown = [re.sub(r"\s*\(.*", "", u).replace("_", " ") for u in prof["unknown"]]
    unknown = sorted(set(unknown) | {"allergies"})
    add("profile", "Not known about this person: " + ", ".join(unknown) + ".")

    # ---- the recipe
    add("recipe", f'The recipe is called "{ci["title"]}".')
    ings = ci["main_ingredients"]
    if ings:
        add("recipe", "Ingredients (as listed): " + ", ".join(ings) + ".")
    d = f["time"]["duration_minutes"]
    add("recipe", f"It takes about {d:.0f} minutes in total.")
    add("recipe", f"Per serving: {n['calories']:.0f} kcal, {n['protein_g']:.0f} g protein, {n['total_fat_g']:.0f} g fat "
                  f"({n['saturated_fat_g']:.0f} g saturated), {n['carbs_g']:.0f} g carbohydrate ({n['sugars_g']:.0f} g sugar, "
                  f"{n['fiber_g']:.0f} g fibre), {n['sodium_mg']:.0f} mg sodium.")
    p = f["nutrition_percentiles"]
    parts = [f"{level(p[k])} in {k}" for k in ("calories", "sodium", "sugar", "saturated fat", "protein", "fiber") if k in p]
    if parts:
        add("recipe", "Compared with other recipes in the dataset, this recipe is " + ", ".join(parts) + ".")
    hl = f["health_levels"]
    add("recipe", f"Nutrition-score levels (higher is healthier): {hl['nutri_level_0_to_4']} out of 4 (Nutri-Score style) "
                  f"and {hl['fsa_level_0_to_8']} out of 8 (FSA style).")

    # ---- how the recipe matches the person
    for chk in f["constraint_checks"]:
        c = chk["constraint"]
        if chk["rule_passes"]:
            how = {"verified": "the ingredient list and the recipe tags both indicate this",
                   "manual": "this was checked by hand"}.get(chk["label_status"],
                                                             "no ingredient in the list conflicts with it, but the recipe carries no explicit tag for it")
            add("match", f"Dietary check ({c}): passes; {how}.")
        else:
            add("match", f"Dietary check ({c}): FAILS; the ingredient list conflicts with it.", must=True)
    tm = f["time"]
    if tm["limit_minutes"]:
        if tm["within_limit"]:
            add("match", f"Time check: {d:.0f} minutes is within the {int(tm['limit_minutes'])}-minute limit.")
        else:
            add("match", f"Time check: {d:.0f} minutes is longer than the {int(tm['limit_minutes'])}-minute limit.", must=True)
    gl = f.get("goal")
    if gl:
        label, nkey, unit = GOAL_NUTRIENT[gl["goal"]]
        base = f"Goal check ({gl['goal']}): {{}}; this recipe's {label} is {level(gl['recipe_percentile'])} compared with other recipes ({n[nkey]:.0f} {unit} per serving)."
        add("match", base.format("met") if gl["meets_goal"] else base.format("NOT met"), must=not gl["meets_goal"])
    if f["matches_liked"]:
        add("match", "The recipe features liked items: " + ", ".join(f["matches_liked"]) + ".")
    elif prof["likes"]:
        add("match", "The recipe features none of the person's listed likes.", must=True)
    if f["contains_disliked"]:
        add("match", "The recipe contains disliked items: " + ", ".join(f["contains_disliked"]) + ".", must=True)
    for ta in ci["tendencies_sorted"][:2]:
        if ta.get("aligned") is not None:
            add("match", f"Compared with the person's usual pattern ({_tendency_phrase(ta['metric'], ta['direction'])}), this recipe is "
                         f"{'in line' if ta['aligned'] else 'not in line'}.")
    return facts


def facts_to_prompt(facts: list[dict]) -> str:
    return "\n".join(f"{x['id']}: {x['text']}" + (" [MUST MENTION]" if x["must_mention"] else "") for x in facts)


In [12]:
# ----------------------------------------------------------------------------- technique 1: template (jinja2, fixed sentences)
_ENV = Environment(undefined=StrictUndefined, trim_blocks=True, lstrip_blocks=True)
_SENT = {
    "open": ["{{ title }} is recommended for you.", "We suggest {{ title }} for you.", "{{ title }} is one option we can offer you."],
    "taste_match_syn": ["It features {{ items }}, which you said you like.", "It includes {{ items }}, which are on your list of favourites."],
    "taste_match_hist": ["It features {{ items }}, and your past reviews suggest you tend to like recipes with these.",
                         "Your past reviews suggest you tend to enjoy recipes featuring {{ items }}, and this one does."],
    "taste_none_syn": ["It does not feature any of the foods you listed as favourites."],
    "taste_none_hist": ["It does not clearly match the ingredients and themes you usually favour."],
    "dislike_syn": ["Note that it contains {{ items }}, which you said you dislike."],
    "dislike_hist": ["Note that it contains {{ items }}, which your past reviews suggest you tend to rate lower."],
    "cons_verified": ["It meets your {{ items }} {{ 'requirements' if plural else 'requirement' }}: the ingredient list and the recipe tags agree."],
    "cons_rule_only": ["No ingredient in the list conflicts with your {{ items }} {{ 'requirements' if plural else 'requirement' }}."],
    "cons_manual": ["It meets your {{ items }} {{ 'requirements' if plural else 'requirement' }} (checked by hand)."],
    "cons_pattern": ["You often choose {{ items }} recipes, and this one fits that pattern."],
    "cons_fail": ["Warning: it does not meet your {{ items }} requirement."],
    "time_ok": ["It takes about {{ d }} minutes, within your {{ limit }}-minute limit."],
    "time_over": ["Note that it takes about {{ d }} minutes, longer than your {{ limit }}-minute limit."],
    "time_free": ["It takes about {{ d }} minutes."],
    "goal_met": ["It supports your goal of {{ goal }}: its {{ nutrient }} is {{ level }} compared with other recipes ({{ val }} {{ unit }} per serving)."],
    "goal_not": ["Trade-off: it does not fit your goal of {{ goal }}, since its {{ nutrient }} is {{ level }} compared with other recipes ({{ val }} {{ unit }} per serving)."],
    "tend_in": ["It matches the usual pattern of your favourites, which tend to have {{ phrase }}."],
    "tend_out": ["It differs from the usual pattern of your favourites, which tend to have {{ phrase }}."],
    "snapshot": ["Per serving it has about {{ cal }} kcal, {{ protein }} g of protein and {{ sodium }} mg of sodium."],
}
_SENT = {k: [_ENV.from_string(v) for v in vs] for k, vs in _SENT.items()}


def _r(name: str, key: str, **kw) -> str:
    vs = _SENT[name]
    i = int(hashlib.md5(f"{key}|{name}".encode()).hexdigest(), 16) % len(vs)
    return vs[i].render(**kw)


def render_template(ci: dict, key: str) -> str:
    f, n, hist = ci["pool_facts"], ci["pool_facts"]["nutrition_per_serving"], ci["profile_source"] == "historical"
    out = [_r("open", key, title=ci["title"])]
    if f["matches_liked"]:
        out.append(_r("taste_match_hist" if hist else "taste_match_syn", key, items=join_and(f["matches_liked"][:3])))
    elif ci["profile_view"]["likes"]:
        out.append(_r("taste_none_hist" if hist else "taste_none_syn", key))
    if f["contains_disliked"]:
        out.append(_r("dislike_hist" if hist else "dislike_syn", key, items=join_and(f["contains_disliked"][:2])))
    groups = {"verified": [], "rule_only": [], "manual": [], "pattern": []}
    for c in f["constraint_checks"]:
        name = c["constraint"]
        if not c["rule_passes"]:
            out.append(_r("cons_fail", key, items=name))
        elif ci["profile_view"]["constraint_status"].get(name) == "observed_pattern":
            groups["pattern"].append(name)
        else:
            groups[c["label_status"] if c["label_status"] in groups else "rule_only"].append(name)
    for g, names in groups.items():
        if names:
            out.append(_r(f"cons_{g}", key, items=join_and(names), plural=len(names) > 1))
    tm = f["time"]
    d = round(tm["duration_minutes"])
    if tm["limit_minutes"]:
        out.append(_r("time_ok" if tm["within_limit"] else "time_over", key, d=d, limit=int(tm["limit_minutes"])))
    else:
        out.append(_r("time_free", key, d=d))
    gl = f.get("goal")
    if gl:
        label, nkey, unit = GOAL_NUTRIENT[gl["goal"]]
        out.append(_r("goal_met" if gl["meets_goal"] else "goal_not", key, goal=gl["goal"], nutrient=label,
                      level=level(gl["recipe_percentile"]), val=round(n[nkey]), unit=unit))
    elif ci["tendencies_sorted"] and ci["tendencies_sorted"][0].get("aligned") is not None:
        t0 = ci["tendencies_sorted"][0]
        out.append(_r("tend_in" if t0["aligned"] else "tend_out", key, phrase=_tendency_phrase(t0["metric"], t0["direction"])))
    out.append(_r("snapshot", key, cal=round(n["calories"]), protein=round(n["protein_g"]), sodium=round(n["sodium_mg"])))
    return " ".join(out)


#### Generating the Explanations
Prompts, case selection (applies `constraint_overrides.csv`), the four techniques, the self-check loop, and `run(args)` - the notebook version of the script's `main()`.
`run(args)` takes the settings object defined in Section 5. Definitions only - nothing runs here.

In [13]:
PROMPT_VERSION = "v1"
TECH_CODE = {"template": "tpl", "direct_llm": "dir", "grounded_llm": "grd", "grounded_check_llm": "chk"}
CLI_TECH = {"template": "template", "direct": "direct_llm", "grounded": "grounded_llm", "check": "grounded_check_llm"}
LO_WORDS, HI_WORDS = 45, 125

# ------------------------------------------------------------------------------ prompts
DIRECT_SYSTEM = "You are a helpful assistant for a food recommendation app."

GROUNDED_SYSTEM = """You write short explanations for a food recommendation system. You explain to one person why one recipe is being recommended to them.

Rules you must follow:
1. Use ONLY the numbered facts provided. Do not add any other information about the person or the recipe, and do not use outside knowledge about the recipe (for example taste, texture, origin or health effects) unless a fact states it.
2. After every sentence, cite the fact numbers that support it in square brackets, for example [F3] or [F2, F5].
3. Facts marked MUST MENTION describe a mismatch or limitation (for example a goal that is not met). State each one honestly as a trade-off, in plain words. Never hide it or turn it into something positive.
4. Facts described as inferred or not confirmed must be phrased cautiously (for example "your past reviews suggest ..."). Never present them as certain.
5. Never assume anything listed under "not known" (for example allergies, budget, family, cooking skill, medical conditions).
6. Use only numbers that appear in the facts.
7. Write one paragraph of 60 to 110 words in plain text, addressed to the person as "you". No lists, headings or bold text."""

NEUTRAL_TONE = "Tone: neutral and informative. Do not use motivational or persuasive language about health."
FRAMED_TONE = ("Tone: frame the explanation around the person's health goal ({goal}). You may be encouraging, but every claim must "
               "still be supported by the facts, and any trade-off must still be stated honestly.")

CHECK_SYSTEM = """You are a strict fact-checker. You are given numbered FACTS and an EXPLANATION written from them.
Break the explanation into its individual factual claims (one checkable statement about the person or the recipe, including any comparison or number). For each claim decide:
- "supported": directly stated by, or a direct restatement of, one or more facts
- "unsupported": not stated by any fact (outside knowledge, invented preferences, assumptions about the person, or vague praise such as "perfect" or "healthy" that no fact supports)
- "contradicted": conflicts with a fact
Return ONLY a JSON object, with no other text, in exactly this form:
{"claims": [{"claim": "...", "verdict": "supported", "fact_ids": ["F3"]}]}"""

REVISE_SYSTEM = GROUNDED_SYSTEM + "\n\nYou are revising a draft that contains problems. Fix every listed problem and keep everything that was correct."


def msgs(system: str, user: str) -> list[dict]:
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


In [14]:
# ------------------------------------------------------------------------------ case selection
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def load_overrides(path) -> dict:
    if not path.exists():
        return {}
    d = pd.read_csv(path)
    bad = set(d["true_status"]) - {"ok", "violates", "ambiguous"}
    if bad:
        raise RuntimeError(f"{path}: true_status must be ok / violates / ambiguous, found {sorted(bad)}")
    return {(int(r.recipe_id), r.constraint): r.true_status for r in d.itertuples()}


def apply_overrides(rec: dict, ov: dict) -> None:
    for ck in rec["facts"]["constraint_checks"]:
        s = ov.get((rec["recipe_id"], ck["constraint"]))
        if s == "ok":
            ck["rule_passes"], ck["label_status"] = True, "manual"
        elif s == "violates":
            ck["rule_passes"], ck["label_status"] = False, "manual"
        elif s == "ambiguous":
            ck["label_status"] = "ambiguous"


def check_state(rec: dict) -> str:
    cks = rec["facts"]["constraint_checks"]
    if any(c["label_status"] == "ambiguous" for c in cks):
        return "ambiguous"
    return "violates" if any(not c["rule_passes"] for c in cks) else "ok"


def select_cases(pool, ov, allow_ambiguous):
    by = defaultdict(list)
    for r in pool:
        apply_overrides(r, ov)
        by[r["profile_id"]].append(r)
    cases, notes = [], []
    for pid, recs in sorted(by.items()):
        chosen = sorted((r for r in recs if r["selected_for_case"]), key=lambda r: r.get("case_slot") or 9)
        final = []
        for r in chosen:
            st = check_state(r)
            if st == "ok" or (st == "ambiguous" and allow_ambiguous):
                r["label_uncertain"] = st == "ambiguous"
                final.append(r)
                continue
            repl = next((c for c in sorted(recs, key=lambda x: x["rank_in_role"]) if c["role"] == "strong" and not c["selected_for_case"]
                         and not c["planned_diagnostic"] and check_state(c) == "ok" and c not in final), None)
            why = f"recipe {r['recipe_id']} ({r['role']}) label is {st}"
            if repl:
                repl["selected_for_case"], repl["case_slot"], repl["replaced_from"] = True, r["case_slot"], f"{r['recipe_id']}:{r['role']}:{st}"
                repl["label_uncertain"] = False
                final.append(repl)
                notes.append(f"{pid}: {why} -> replaced by strong candidate {repl['recipe_id']}")
            else:
                notes.append(f"{pid}: {why} and no replacement available -> case dropped")
        cases += final
    return cases, notes


def build_case_input(rec, prof, recipe) -> dict:
    f = rec["facts"]
    tz = {t["metric"]: t for t in prof.get("nutrition_tendencies", [])}
    tsorted = [{"metric": ta["metric"], "direction": ta["direction"], "aligned": ta["aligned"],
                "confidence": tz.get(ta["metric"], {}).get("confidence", "medium"), "z": tz.get(ta["metric"], {}).get("z", 0)}
               for ta in sorted(f["tendency_alignment"], key=lambda x: -abs(tz.get(x["metric"], {}).get("z", 0)))]
    hg, tb = prof["health_goal"], prof["time_budget_minutes"]
    view = {
        "n_reviews": (prof.get("history") or {}).get("n_reviews"),
        "constraints": [{"constraint": c["constraint"], "status": c["status"]} for c in prof["dietary_constraints"]],
        "constraint_status": {c["constraint"]: c["status"] for c in prof["dietary_constraints"]},
        "goal": {"value": hg["value"], "status": hg["status"]},
        "time": {"value": tb.get("value"), "status": tb.get("status"), "typical_minutes": tb.get("typical_minutes")},
        "likes": [x["item"] for x in prof["taste"]["likes"][:5]],
        "dislikes": [x["item"] for x in prof["taste"]["dislikes"][:3]],
        "tension": prof.get("tension"), "unknown": prof.get("unknown_fields", []),
    }
    ci = {
        "case_key": f"{rec['profile_id']}-r{rec['case_slot']}", "profile_id": rec["profile_id"], "profile_source": rec["profile_source"],
        "profile_label": prof["label"], "profile_text": scrub_text(prof["profile_text"]), "recipe_id": rec["recipe_id"],
        "title": scrub_text(recipe["title"]), "role": rec["role"], "case_slot": rec["case_slot"],
        "replaced_from": rec.get("replaced_from"), "label_uncertain": bool(rec.get("label_uncertain")),
        "pool_facts": f, "profile_view": view, "tendencies_sorted": tsorted,
        "main_ingredients": main_ingredients(recipe), "ingredient_lines": ingredient_lines(recipe),
        "goal_key": f["goal"]["goal"] if f.get("goal") else None,
    }
    ci["facts"] = build_facts(ci)
    return ci


def order_cases(cis, limit):
    """Whole profiles at a time, alternating historical / synthetic, so a pilot (--limit) covers both sources and both slots."""
    by = defaultdict(list)
    for c in cis:
        by[c["profile_id"]].append(c)
    hist = [sorted(v, key=lambda c: c["case_slot"]) for k, v in sorted(by.items()) if v[0]["profile_source"] == "historical"]
    syn = [sorted(v, key=lambda c: c["case_slot"]) for k, v in sorted(by.items()) if v[0]["profile_source"] == "synthetic"]
    out = [c for pair in itertools.zip_longest(hist, syn) for grp in pair if grp for c in grp]
    return out[:limit] if limit else out


In [15]:
# ------------------------------------------------------------------------------ prompts per technique
def direct_messages(ci):
    n, tm = ci["pool_facts"]["nutrition_per_serving"], ci["pool_facts"]["time"]["duration_minutes"]
    ings = "\n".join(f"- {x}" for x in ci["ingredient_lines"])
    user = (f"PERSON\n{ci['profile_text']}\n\nRECIPE\nTitle: {ci['title']}\nIngredients:\n{ings}\nReady in about {tm:.0f} minutes.\n"
            f"Nutrition per serving: {n['calories']:.0f} kcal, {n['protein_g']:.0f} g protein, {n['total_fat_g']:.0f} g fat "
            f"({n['saturated_fat_g']:.0f} g saturated), {n['carbs_g']:.0f} g carbohydrate ({n['sugars_g']:.0f} g sugar, "
            f"{n['fiber_g']:.0f} g fibre), {n['sodium_mg']:.0f} mg sodium.\n\n"
            'Write a short explanation (60 to 110 words, one paragraph, plain text, no lists or headings) addressed to the person as "you", '
            "explaining why this recipe is being recommended to them.")
    return msgs(DIRECT_SYSTEM, user)


def grounded_messages(ci, framing):
    tone = NEUTRAL_TONE if framing == "neutral" else FRAMED_TONE.format(goal=ci["goal_key"])
    return msgs(GROUNDED_SYSTEM, f"FACTS\n{facts_to_prompt(ci['facts'])}\n\n{tone}\n\nWrite the explanation now.")


def parse_claims(text):
    t = text.replace("```json", "").replace("```", "")
    m = re.search(r"\{.*\}", t, flags=re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    cl = d.get("claims") if isinstance(d, dict) else None
    if not isinstance(cl, list):
        return None
    out = [{"claim": str(c.get("claim", "")), "verdict": str(c["verdict"]).lower().strip(), "fact_ids": c.get("fact_ids", [])}
           for c in cl if isinstance(c, dict) and "verdict" in c]
    return out or None


In [16]:
class Gen:
    """Holds the client and settings; every LLM call goes through here."""

    def __init__(self, client, args, audit_client=None):
        self.client, self.a = client, args
        self.audit_client = audit_client or client

    def write(self, messages, task, meta):
        r = self.client.chat(messages, task=task, temperature=self.a.temperature, max_tokens=self.a.max_tokens, meta=meta)
        text, w = r.text, word_count(clean_text(r.text))
        if w < LO_WORDS or w > HI_WORDS:                                  # one length-repair attempt
            fix = messages + [{"role": "assistant", "content": text},
                              {"role": "user", "content": f"That answer has {w} words. Rewrite it in 60 to 110 words, following the same rules, "
                                                          "and return only the explanation."}]
            text = self.client.chat(fix, task="shorten", temperature=self.a.temperature, max_tokens=self.a.max_tokens,
                                    meta={**meta, "text": text}).text
        return text

    def audit(self, ci, raw):
        facts, clean = ci["facts"], clean_text(raw)
        claims = None
        for attempt in range(2):
            tail = "" if attempt == 0 else "\n\nYour previous reply was not valid JSON. Return ONLY the JSON object."
            r = self.audit_client.chat(msgs(CHECK_SYSTEM, f"FACTS\n{facts_to_prompt(facts)}\n\nEXPLANATION\n{clean}\n\nReturn the JSON now.{tail}"),
                                 task="check", temperature=0.0, max_tokens=self.a.check_max_tokens, meta={"text": clean, "facts": facts})
            claims = parse_claims(r.text)
            if claims:
                break
        check_ok = bool(claims)
        issues = [{"type": c["verdict"], "detail": c["claim"]} for c in (claims or []) if c["verdict"] in ("unsupported", "contradicted")]
        issues += [{"type": "number_not_in_facts", "detail": n} for n in numeric_mismatches(raw, facts)]
        cited = cited_ids(raw)
        missing = [f["id"] for f in facts if f["must_mention"] and f["id"] not in cited]
        issues += [{"type": "missing_tradeoff", "detail": fid} for fid in missing]
        return {
            "issues": issues,
            "claims": claims or [],
            "check_ok": check_ok,
            "missing_ids": missing,
            "auditor": {
                "provider": self.audit_client.provider,
                "model": self.audit_client.model,
            },
        }

    def revise(self, ci, raw, a):
        by = {f["id"]: f["text"] for f in ci["facts"]}
        lines = []
        for i in a["issues"]:
            t = i["type"]
            if t in ("unsupported", "contradicted"):
                lines.append(f'- {t.capitalize()} claim - remove or correct it: "{i["detail"]}"')
            elif t == "number_not_in_facts":
                lines.append(f"- The number {i['detail']} does not appear in the facts; use only numbers from the facts, or remove it.")
            elif t == "missing_tradeoff":
                lines.append(f'- Missing trade-off: you must state {i["detail"]} ("{by[i["detail"]]}") honestly.')
        user = (f"FACTS\n{facts_to_prompt(ci['facts'])}\n\nCURRENT DRAFT\n{raw}\n\nPROBLEMS TO FIX\n" + "\n".join(lines) +
                "\n\nRewrite the explanation so that every problem is fixed. Keep the rules. Return only the corrected explanation, with fact citations.")
        return self.write(msgs(REVISE_SYSTEM, user), "revise", {"text": raw, "facts": ci["facts"], "missing_ids": a["missing_ids"], "title": ci["title"], "case": ci["case_key"]})

    def check_and_revise(self, ci, draft):
        text, trail, first = draft, [], None
        best_text, best_audit, best_iteration = draft, None, None
        for it in range(self.a.max_revisions + 1):
            a = self.audit(ci, text)
            trail.append({"iteration": it, "auditor": a["auditor"], "n_claims": len(a["claims"]), "check_ok": a["check_ok"], "issues": a["issues"]})
            first = first or a
            # Keep the least-flagged version; ties preserve the earlier (simpler) draft.
            if best_audit is None or len(a["issues"]) < len(best_audit["issues"]):
                best_text, best_audit, best_iteration = text, a, it
            if not a["issues"] or it == self.a.max_revisions:
                break
            text = self.revise(ci, text, a)
        return best_text, {"n_revisions": len(trail) - 1, "initial_issue_count": len(first["issues"]), "final_issue_count": len(best_audit["issues"]),"selected_iteration": best_iteration,
                            "auditor": best_audit["auditor"], "check_failed": not all(t["check_ok"] for t in trail), "claims_initial": first["claims"], "claims_final": best_audit["claims"], "trail": trail}


In [17]:
def new_record(ci, tech, framing, key, gen_info):
    return {"key": key, "case_key": ci["case_key"], "profile_id": ci["profile_id"], "profile_source": ci["profile_source"],
            "recipe_id": ci["recipe_id"], "case_slot": ci["case_slot"], "role": ci["role"], "technique": tech, "framing": framing,
            "prompt_version": PROMPT_VERSION, "provider": gen_info[0], "model": gen_info[1],
            "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}


In [18]:
def run(args):
    techs = [CLI_TECH[t.strip()] for t in args.techniques.split(",") if t.strip()]

    # ---- cases
    pool = read_jsonl(PROC_DIR / "recommendation_pool.jsonl")
    profiles = {}
    for name in ("profiles_historical", "profiles_synthetic"):
        for p in read_jsonl(PROC_DIR / f"{name}.jsonl"):
            profiles[p["profile_id"]] = p
    ov = load_overrides(__import__("pathlib").Path(args.overrides))
    recs, notes = select_cases(pool, ov, args.allow_ambiguous)
    print(f"{len(ov)} label overrides applied" if ov else "no constraint_overrides.csv found (labels used as-is)")
    for n in notes:
        print("  NOTE", n)
    ids = {r["recipe_id"] for r in recs}
    cat = read_table("catalog", usecols=["new_recipe_id", "title", "ingredient_names", "ingredients_display"], filter_col="new_recipe_id", filter_values=ids)
    cat = cat.drop_duplicates("new_recipe_id").set_index("new_recipe_id")
    cis = []
    for r in recs:
        row = cat.loc[r["recipe_id"]]
        recipe = {"title": row["title"], "ingredient_names": as_list(row["ingredient_names"]), "ingredients_display": as_list(row["ingredients_display"])}
        cis.append(build_case_input(r, profiles[r["profile_id"]], recipe))
    cis = order_cases(cis, args.limit)
    with open(PROC_DIR / "case_inputs.jsonl", "w", encoding="utf-8") as f:
        for ci in cis:
            f.write(json.dumps(ci, ensure_ascii=False, default=str) + "\n")
    per_profile = defaultdict(int)
    for c in cis:
        per_profile[c["profile_id"]] += 1
    short = [p for p, n in per_profile.items() if n < 2]
    print(f"{len(cis)} cases over {len(per_profile)} profiles ({sum(c['profile_source'] == 'historical' for c in cis)} historical, "
          f"{sum(c['profile_source'] == 'synthetic' for c in cis)} synthetic)" + (f" | profiles with <2 cases: {short}" if short else ""))
    uncertain = [c["case_key"] for c in cis if c["label_uncertain"]]
    if uncertain:
        print("  WARNING cases kept with an ambiguous dietary label:", uncertain)

    framed_keys = set()
    if args.framed_n > 0 and "grounded_llm" in techs:
        elig = sorted((c for c in cis if c["goal_key"] and c["profile_view"]["goal"]["status"] == "stated" and c["case_slot"] == 1), key=lambda c: c["profile_id"])
        framed_keys = {c["case_key"] for c in elig[: args.framed_n]}

    n_llm = sum(t != "template" for t in techs)
    est_calls = len(cis) * (("direct_llm" in techs) + ("grounded_llm" in techs) + (3 if "grounded_check_llm" in techs else 0)) + len(framed_keys)
    print(f"techniques: {', '.join(techs)} | framed extras: {len(framed_keys)} | estimated LLM calls: ~{est_calls}" if n_llm else f"techniques: {', '.join(techs)}")

    # ---- dry run
    if args.dry_run:
        lines = []
        for ci in cis[: args.preview_n]:
            lines += [f"{'=' * 100}\nCASE {ci['case_key']}  role={ci['role']}  profile={ci['profile_label']}\n{'=' * 100}",
                      "--- FACT LIST ---", facts_to_prompt(ci["facts"]), "--- TEMPLATE OUTPUT ---", render_template(ci, f"{ci['case_key']}-tpl"),
                      "--- DIRECT PROMPT ---", direct_messages(ci)[1]["content"], "--- GROUNDED PROMPT (user part) ---", grounded_messages(ci, "neutral")[1]["content"], ""]
        (PROC_DIR / "prompt_preview.txt").write_text("\n".join(lines), encoding="utf-8")
        tw = [word_count(render_template(c, f"{c['case_key']}-tpl")) for c in cis]
        print(f"dry run: wrote processed/prompt_preview.txt | template words min/mean/max = {min(tw)}/{sum(tw) / len(tw):.0f}/{max(tw)}")
        return

    # ---- client + resume
    extra = json.loads(args.extra) if args.extra else None
    client_kw = {"rpm": args.rpm, "tpm": args.tpm, "extra": extra}
    client = get_client(args.provider, args.model, **client_kw if args.provider != "mock" else {})
    if args.provider == "mock":
        # Offline tests use the same mock for both roles; real runs must use a distinct auditor.
        audit_client = client
    else:
        if args.audit_provider == "mock":
            raise ValueError("For a real generation run, audit_provider must be a real model provider, not mock.")
        if (args.audit_provider, args.audit_model) == (args.provider, args.model):
            raise ValueError("The auditor must use a different provider/model from the writer.")
        audit_client = get_client(args.audit_provider, args.audit_model,
                                  **(client_kw if args.audit_provider != "mock" else {}))
    gen_info = (client.provider, client.model)
    audit_info = (audit_client.provider, audit_client.model)
    out_path = PROC_DIR / "explanations.jsonl"
    done: dict = {}
    if out_path.exists() and not args.fresh:
        for r in read_jsonl(out_path):
            done[r["key"]] = r
        done = {k: r for k, r in done.items() if not r.get("error")}
        mixed = {(r["provider"], r["model"]) for r in done.values() if r["technique"] != "template"} - {gen_info}
        if mixed:
            raise RuntimeError(f"existing explanations were generated by {sorted(mixed)} but this run uses {gen_info}. "
                     "All LLM techniques must use ONE generator: use the same model, or pass --fresh to start over.")
        with open(out_path, "w", encoding="utf-8") as f:
            for r in done.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    elif out_path.exists():
        out_path.unlink()
    if done:
        print(f"resuming: {len(done)} explanations already done")

    gen = Gen(client, args, audit_client=audit_client)
    drafts = {(r["case_key"], r["framing"]): r["text_raw"] for r in done.values() if r["technique"] == "grounded_llm"}
    errors, quota_stop = [], False
    t_start = time.time()
    fout = open(out_path, "a", encoding="utf-8")

    def run_one(ci, tech, framing):
        key = f"{ci['case_key']}-{TECH_CODE[tech]}" + ("-framed" if framing == "goal_framed" else "")
        if key in done:
            return
        rec = new_record(ci, tech, framing, key, ("jinja2", "template") if tech == "template" else gen_info)
        meta = {"title": ci["title"], "facts": ci["facts"], "case": key, "minutes": round(ci["pool_facts"]["time"]["duration_minutes"]),
                "ingredients": ci["main_ingredients"]}
        try:
            if tech == "template":
                raw = render_template(ci, key)
            elif tech == "direct_llm":
                raw = gen.write(direct_messages(ci), "explain_direct", meta)
            elif tech == "grounded_llm":
                raw = gen.write(grounded_messages(ci, framing), "explain_grounded", meta)
                drafts[(ci["case_key"], framing)] = raw
            else:
                draft = drafts.get((ci["case_key"], "neutral"))
                if draft is None:
                    draft = gen.write(grounded_messages(ci, "neutral"), "explain_grounded", meta)
                    drafts[(ci["case_key"], "neutral")] = draft
                raw, audit = gen.check_and_revise(ci, draft)
                rec.update({"audit": audit, "draft_raw": draft, "changed_vs_grounded": clean_text(raw) != clean_text(draft)})
            rec["text_raw"], rec["text"] = raw, clean_text(raw)
            rec["words"] = word_count(rec["text"])
            if tech != "template":
                rec["numeric_mismatches"] = numeric_mismatches(raw, ci["facts"])
            if tech in ("grounded_llm", "grounded_check_llm"):
                rec["cited_fact_ids"] = sorted(cited_ids(raw))
        except QuotaExhausted:
            raise
        except Exception as e:  # noqa: BLE001 - keep going, record the failure, retry on the next run
            rec["error"] = f"{type(e).__name__}: {str(e)[:300]}"
            errors.append((key, rec["error"]))
            print(f"  ERROR {key}: {rec['error']}")
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fout.flush()
        if "error" not in rec:
            done[key] = rec

    try:
        for i, ci in enumerate(cis, 1):
            for tech in techs:
                run_one(ci, tech, "neutral")
            if ci["case_key"] in framed_keys:
                run_one(ci, "grounded_llm", "goal_framed")
            if i % 5 == 0 or i == len(cis):
                print(f"  {i}/{len(cis)} cases | {len(done)} explanations | {time.time() - t_start:.0f}s | {client.stats()['api_calls']} API calls, "
                      f"{client.stats()['cache_hits']} cache hits")
    except QuotaExhausted as e:
        quota_stop = True
        print(f"\nSTOPPED: quota exhausted ({e}).\nProgress is saved. Re-run the same command later (or tomorrow) and it resumes.")
    finally:
        fout.close()

    # ---- summary + report
    recs_out = [r for r in done.values()]
    df = pd.DataFrame([{k: r.get(k) for k in ("key", "case_key", "profile_id", "profile_source", "case_slot", "role", "technique", "framing", "words")}
                       | {"n_numeric_mismatch": len(r.get("numeric_mismatches", [])),
                          "n_revisions": (r.get("audit") or {}).get("n_revisions"), "changed_vs_grounded": r.get("changed_vs_grounded"),
                          "text": r["text"]} for r in recs_out])
    df.to_csv(PROC_DIR / "explanations_summary.csv", index=False)
    rep = {"prompt_version": PROMPT_VERSION, "generator": {"provider": gen_info[0], "model": gen_info[1]}, "settings": {
               "temperature": args.temperature, "max_tokens": args.max_tokens, "max_revisions": args.max_revisions, "framed_n": args.framed_n},
           "auditor": {"provider": audit_info[0], "model": audit_info[1]},
           "cases": len(cis), "explanations_done": len(df), "main_480_done": int((df["framing"] == "neutral").sum()) if len(df) else 0,
           "target_main": len(cis) * len(techs), "errors": errors, "stopped_on_quota": quota_stop, "llm_usage": client.stats(), "audit_llm_usage": audit_client.stats()}
    if len(df):
        rep["by_technique"] = {t: {"n": int(len(g)), "words_mean": round(g["words"].mean(), 1), "words_min": int(g["words"].min()), "words_max": int(g["words"].max()),
                                   "share_with_number_not_in_facts": round(float((g["n_numeric_mismatch"] > 0).mean()), 3)}
                               for t, g in df.groupby("technique")}
        chk = df[df["technique"] == "grounded_check_llm"]
        if len(chk):
            rep["check_revise"] = {"share_revised": round(float((chk["n_revisions"] > 0).mean()), 3), "mean_revisions": round(float(chk["n_revisions"].mean()), 2),
                                   "share_text_changed_vs_grounded": round(float(chk["changed_vs_grounded"].mean()), 3),
                                   "check_failed": int(sum((r.get("audit") or {}).get("check_failed", False) for r in recs_out if r["technique"] == "grounded_check_llm")),
                                   "unresolved_after_revisions": int(sum((r.get("audit") or {}).get("final_issue_count", 0) > 0 for r in recs_out if r["technique"] == "grounded_check_llm"))}
    (PROC_DIR / "generation_report.json").write_text(json.dumps(rep, indent=2), encoding="utf-8")
    print("\n" + json.dumps({k: rep[k] for k in ("generator", "cases", "explanations_done", "target_main", "stopped_on_quota", "llm_usage")}, indent=2))
    for t, v in rep.get("by_technique", {}).items():
        print(f"  {t:20s} n={v['n']:4d} words {v['words_min']}-{v['words_max']} (mean {v['words_mean']})  numbers-not-in-facts: {v['share_with_number_not_in_facts']:.0%}")
    if "check_revise" in rep:
        print("  check/revise:", rep["check_revise"])
    if errors:
        print(f"{len(errors)} explanations failed; re-run to retry only those.")
    return rep


In [19]:
# Run this cell after manually filling manual_ok in processed/constraint_check_sheet_filled.csv.
# Accepted values are exactly: ok, violates, ambiguous (case-insensitive).
sheet_path = PROC_DIR / "constraint_check_sheet.csv"
overrides_path = PROC_DIR / "constraint_overrides.csv"
if not sheet_path.exists():
    raise FileNotFoundError(f"Missing {sheet_path}. Run Notebook 4 first to generate the check sheet and then fill the manual_ok column.")

sheet = pd.read_csv(sheet_path, dtype={"constraint": "string", "manual_ok": "string"})
required = {"recipe_id", "constraint", "manual_ok"}
missing = required - set(sheet.columns)
if missing:
    raise ValueError(f"{sheet_path} is missing columns: {sorted(missing)}")

manual = sheet["manual_ok"].fillna("").astype(str).str.strip().str.lower()
allowed = {"ok", "violates", "ambiguous"}
unexpected = sorted(set(manual) - {""} - allowed)
if unexpected:
    raise ValueError(f"Invalid manual_ok labels {unexpected}; use only ok / violates / ambiguous.")

out = sheet.loc[manual.ne(""), ["recipe_id", "constraint"]].copy()
out["true_status"] = manual.loc[manual.ne("")].to_numpy()
if out.empty:
    raise RuntimeError(
        f"No manual labels found in {sheet_path}. Fill manual_ok before running Notebook 5; "
        "otherwise all constraint checks remain keyword/rule-based."
    )
out["recipe_id"] = pd.to_numeric(out["recipe_id"], errors="raise").astype(int)
out["constraint"] = out["constraint"].astype(str).str.strip()

conflicts = out.groupby(["recipe_id", "constraint"])["true_status"].nunique()
conflicts = conflicts[conflicts > 1]
if not conflicts.empty:
    raise ValueError(f"Conflicting manual labels for recipe_id/constraint pairs: {list(conflicts.index)}")
out = out.drop_duplicates(["recipe_id", "constraint"]).sort_values(["recipe_id", "constraint"])
out = out[["recipe_id", "constraint", "true_status"]]
out.to_csv(overrides_path, index=False)
print(f"Saved {len(out)} manual overrides to {overrides_path}")
print(out["true_status"].value_counts().to_dict())

Saved 50 manual overrides to C:\HP Data\University-Masters\Semester_3\P1_Natural Language Processing\Lab\Project\FoodJudgeBench\processed\constraint_overrides.csv
{'ok': 26, 'violates': 14, 'ambiguous': 10}


#### Settings
Edit this cell, then run it. These are the same options as the script's command-line flags.

| Setting | Meaning |
|---|---|
| `provider`, `model` | `ollama` and the tag from `ollama list` (e.g. `llama3.1:8b`). Keep the **same model for the whole run**. |
| `techniques` | which of `template,direct,grounded,check` to generate |
| `limit` | only the first N cases (whole profiles, alternating historical/synthetic); `None` = all 120 |
| `framed_n` | extra goal-framed grounded explanations (0 = none) |
| `max_revisions` | max revise rounds in the self-check technique |
| `temperature`, `max_tokens`, `check_max_tokens` | writer temperature / writer token budget / audit token budget |
| `allow_ambiguous` | keep cases whose diet label is marked ambiguous in `constraint_overrides.csv` |
| `dry_run` | build facts and prompts only, no model calls |
| `fresh` | discard `explanations.jsonl` and start over (the cache keeps unchanged prompts free) |

In [20]:
PROVIDER = "ollama"          # "ollama" | "groq" | "gemini" | ... | "mock" (offline fake, for testing the notebook)
MODEL = "llama3.1:8b"        # must match `ollama list`; ignored by "mock"

AUDIT_PROVIDER = "ollama"    # Keep the auditor separately configurable.
AUDIT_MODEL = "qwen2.5:14b"

DEFAULTS = dict(
    provider=PROVIDER, model=MODEL, 
    audit_provider=AUDIT_PROVIDER, audit_model=AUDIT_MODEL,
    extra=None, rpm=None, tpm=None,
    techniques="template,direct,grounded,check",
    limit=None, framed_n=20, max_revisions=2,
    temperature=0.4, max_tokens=1200, check_max_tokens=1800,
    overrides=str(PROC_DIR / "constraint_overrides.csv"),
    allow_ambiguous=False, dry_run=False, fresh=False, preview_n=3,
)


def make_args(**changes) -> SimpleNamespace:
    """Settings object for run(); pass keyword changes, e.g. make_args(limit=6)."""
    bad = set(changes) - set(DEFAULTS)
    if bad:
        raise ValueError(f"unknown setting(s): {sorted(bad)}")
    return SimpleNamespace(**{**DEFAULTS, **changes})


print({k: v for k, v in DEFAULTS.items() if k != "overrides"})

{'provider': 'ollama', 'model': 'llama3.1:8b', 'audit_provider': 'ollama', 'audit_model': 'qwen2.5:14b', 'extra': None, 'rpm': None, 'tpm': None, 'techniques': 'template,direct,grounded,check', 'limit': None, 'framed_n': 20, 'max_revisions': 2, 'temperature': 0.4, 'max_tokens': 1200, 'check_max_tokens': 1800, 'allow_ambiguous': False, 'dry_run': False, 'fresh': False, 'preview_n': 3}


#### Testing the connection (Ollama)
Both lines must succeed before you continue. The first checks that the Ollama server is running and the model is installed; the second sends one tiny request.
If the first fails: start Ollama (Windows/Mac: open the app; Linux: `ollama serve`). If the model is missing: `ollama pull llama3.1:8b` in a terminal.
The first request after starting Ollama can take 10-60 s while the model loads into memory.

In [21]:
import ollama
ollama.pull(MODEL)
ollama.pull(AUDIT_MODEL)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [22]:
if PROVIDER == "mock":
    print("mock provider: no model connection test needed")
else:
    if (AUDIT_PROVIDER, AUDIT_MODEL) == (PROVIDER, MODEL):
        raise ValueError("The auditor must use a different provider/model from the writer.")
    if PROVIDER == "ollama":
        assert ollama_check(MODEL), "Writer model is not available in Ollama."
    ping(PROVIDER, MODEL)
    if AUDIT_PROVIDER == "ollama":
        assert ollama_check(AUDIT_MODEL), "Auditor model is not available in Ollama."
    if AUDIT_PROVIDER != "mock":
        ping(AUDIT_PROVIDER, AUDIT_MODEL)

Ollama is running at http://localhost:11434. Installed models: ['qwen2.5:14b', 'llama3.1:8b', 'qwen2.5:7b']
ollama / llama3.1:8b: reply='Done' in 12.2s usage={'prompt_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens': 2, 'total_tokens': 19}
Ollama is running at http://localhost:11434. Installed models: ['qwen2.5:14b', 'llama3.1:8b', 'qwen2.5:7b']
ollama / qwen2.5:14b: reply='Ok' in 10.8s usage={'prompt_tokens': 36, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens': 2, 'total_tokens': 38}


#### Dry run (no model calls)
Builds the cases, the fact lists, the template explanations and the prompts. Prints how many cases/calls the full run would need and writes `processed/prompt_preview.txt` - **open it and read the fact lists and prompts**.

In [23]:
run(make_args(dry_run=True, limit=4))

50 label overrides applied
  NOTE S006: recipe 328264 (strong) label is ambiguous -> replaced by strong candidate 450187
  NOTE S008: recipe 88190 (weak_goal) label is ambiguous -> replaced by strong candidate 332653
  NOTE S010: recipe 11351 (strong) label is ambiguous -> replaced by strong candidate 173252
  NOTE S011: recipe 156094 (weak_goal) label is ambiguous -> replaced by strong candidate 259733
  NOTE S016: recipe 429015 (weak_taste) label is ambiguous -> replaced by strong candidate 130164
  NOTE S021: recipe 332711 (strong) label is violates -> replaced by strong candidate 234471
  NOTE S022: recipe 247296 (strong) label is violates -> replaced by strong candidate 120701
  NOTE S026: recipe 328264 (strong) label is ambiguous -> replaced by strong candidate 482334
4 cases over 2 profiles (2 historical, 2 synthetic)
techniques: template, direct_llm, grounded_llm, grounded_check_llm | framed extras: 0 | estimated LLM calls: ~20
dry run: wrote processed/prompt_preview.txt | temp

In [24]:
print((PROC_DIR / "prompt_preview.txt").read_text(encoding="utf-8")[:6000])

CASE H001-r1  role=strong  profile=rich/no_signal
--- FACT LIST ---
F1: This person's profile is inferred from 173 past recipe reviews; nothing in it was confirmed by the person.
F2: No health goal is known for this person.
F3: No time limit is known; recipes this person liked typically take about 45 minutes.
F4: Past reviews suggest the person tends to like recipes featuring: hand-formed-cookies, cookies-and-brownies, desserts, icing sugar, almond extract (inferred pattern).
F5: Recipes this person liked tend to have lower fiber than typical recipes (medium confidence, inferred).
F6: Recipes this person liked tend to have higher saturated fat than typical recipes (medium confidence, inferred).
F7: Not known about this person: allergies, budget, confirmed dietary restrictions, dislikes, health goal, time budget.
F8: The recipe is called "Sweetslady's Chocolate Peanut Butter Brownie Trifle".
F9: Ingredients (as listed): granulated sugar, baking cocoa, all-purpose flour, baking powder, s

In [25]:
# Add manual-check rows for final cases missing from the sheet.
case_inputs = read_jsonl(PROC_DIR / "case_inputs.jsonl")

recipe_ids = {int(ci["recipe_id"]) for ci in case_inputs}

cat = read_table(
    "catalog",
    usecols=[
        "new_recipe_id", "title",
        "ingredient_names", "ingredients_display",
    ],
    filter_col="new_recipe_id",
    filter_values=recipe_ids,
)

cat = (
    cat.drop_duplicates("new_recipe_id")
       .set_index("new_recipe_id")
)

sheet_path = PROC_DIR / "constraint_check_sheet.csv"
sheet = pd.read_csv(sheet_path)

if "manual_ok" not in sheet.columns:
    sheet["manual_ok"] = ""

sheet["recipe_id"] = pd.to_numeric(
    sheet["recipe_id"], errors="raise"
).astype(int)
sheet["constraint"] = sheet["constraint"].astype(str).str.strip()

existing = set(
    zip(sheet["recipe_id"], sheet["constraint"])
)

new_rows = []

for ci in case_inputs:
    rid = int(ci["recipe_id"])

    for check in ci["pool_facts"].get("constraint_checks", []):
        constraint = check["constraint"]
        key = (rid, constraint)

        if key in existing:
            continue

        recipe = cat.loc[rid]
        ingredients = (
            as_list(recipe["ingredients_display"])
            or as_list(recipe["ingredient_names"])
        )

        new_rows.append({
            "profile_id": ci["profile_id"],
            "recipe_id": rid,
            "title": ci["title"],
            "constraint": constraint,
            "claimed": (
                "OK" if check["rule_passes"] else "VIOLATES"
            ),
            "label_status": check["label_status"],
            "ingredients": " ; ".join(map(str, ingredients)),
            "manual_ok": "",
        })

        existing.add(key)

if new_rows:
    sheet = pd.concat(
        [sheet, pd.DataFrame(new_rows)],
        ignore_index=True,
    )
    sheet.to_csv(sheet_path, index=False)

print(f"Added {len(new_rows)} new recipe-constraint checks.")

print(
    sheet.loc[
        sheet["manual_ok"].fillna("").astype(str).str.strip().eq(""),
        ["profile_id", "recipe_id", "title", "constraint", "ingredients"],
    ].to_string(index=False)
)

Added 0 new recipe-constraint checks.
Empty DataFrame
Columns: [profile_id, recipe_id, title, constraint, ingredients]
Index: []


If the above helper  cell adds new recipe-constraints pair, then before running the next cells, fill up the missing manual_check column in constraint_check_sheet.csv and generate a new constraint_overrides.csv file (helper cell available a few cells above). If no new pairs are added, next cells can be run directly.

#### Pilot (6 cases x 4 techniques)
Real model calls on a few cases. Expect a few minutes with a local 8B model. Read the results before the full run. (Results are kept: the full run in Section 9 continues from them. Nothing is deleted unless you set `fresh=True`.)

In [26]:
report = run(make_args(limit=6))

50 label overrides applied
  NOTE S006: recipe 328264 (strong) label is ambiguous -> replaced by strong candidate 450187
  NOTE S008: recipe 88190 (weak_goal) label is ambiguous -> replaced by strong candidate 332653
  NOTE S010: recipe 11351 (strong) label is ambiguous -> replaced by strong candidate 173252
  NOTE S011: recipe 156094 (weak_goal) label is ambiguous -> replaced by strong candidate 259733
  NOTE S016: recipe 429015 (weak_taste) label is ambiguous -> replaced by strong candidate 130164
  NOTE S021: recipe 332711 (strong) label is violates -> replaced by strong candidate 234471
  NOTE S022: recipe 247296 (strong) label is violates -> replaced by strong candidate 120701
  NOTE S026: recipe 328264 (strong) label is ambiguous -> replaced by strong candidate 482334
6 cases over 3 profiles (4 historical, 2 synthetic)
techniques: template, direct_llm, grounded_llm, grounded_check_llm | framed extras: 0 | estimated LLM calls: ~30
  5/6 cases | 20 explanations | 482s | 13 API call

In [27]:
pilot = pd.read_csv(PROC_DIR / "explanations_summary.csv")
for _, r in pilot[pilot["case_key"] == pilot["case_key"].iloc[0]].iterrows():
    print(f"--- {r['technique']} ({r['framing']}, {r['words']} words) ---\n{r['text']}\n")

--- template (neutral, 68 words) ---
Sweetslady's Chocolate Peanut Butter Brownie Trifle is one option we can offer you. It features desserts, peanut butter and vanilla, and your past reviews suggest you tend to like recipes with these. It takes about 60 minutes. It differs from the usual pattern of your favourites, which tend to have lower fiber. Per serving it has about 508 kcal, 8 g of protein and 272 mg of sodium.

--- direct_llm (neutral, 82 words) ---
I think you'll love Sweetslady's Chocolate Peanut Butter Brownie Trifle! Given your fondness for cookies and brownies, this recipe combines the best of both worlds in a layered dessert. The addition of peanut butter and cream cheese gives it a rich and creamy texture, while the chocolate syrup adds an extra indulgent touch. With its high calorie and sugar content, this treat is sure to satisfy your sweet tooth, and the use of powdered sugar and icing sugar aligns with your preferences.

--- grounded_llm (neutral, 101 words) ---
Base

#### Full run (120 cases x 4 techniques)
Takes a long time on a local model (hours). **You can stop the cell at any time** (Stop button) - every finished explanation is already saved; run the cell again to continue where it stopped. Keep the same settings (same model) between runs.
Use `fresh=True` only when you want to throw everything away (e.g. after changing profiles).

In [28]:
report = run(make_args())

50 label overrides applied
  NOTE S006: recipe 328264 (strong) label is ambiguous -> replaced by strong candidate 450187
  NOTE S008: recipe 88190 (weak_goal) label is ambiguous -> replaced by strong candidate 332653
  NOTE S010: recipe 11351 (strong) label is ambiguous -> replaced by strong candidate 173252
  NOTE S011: recipe 156094 (weak_goal) label is ambiguous -> replaced by strong candidate 259733
  NOTE S016: recipe 429015 (weak_taste) label is ambiguous -> replaced by strong candidate 130164
  NOTE S021: recipe 332711 (strong) label is violates -> replaced by strong candidate 234471
  NOTE S022: recipe 247296 (strong) label is violates -> replaced by strong candidate 120701
  NOTE S026: recipe 328264 (strong) label is ambiguous -> replaced by strong candidate 482334
120 cases over 60 profiles (60 historical, 60 synthetic)
techniques: template, direct_llm, grounded_llm, grounded_check_llm | framed extras: 17 | estimated LLM calls: ~617
resuming: 24 explanations already done
  5/

#### Inspecting the results
Outputs in `processed/`: `case_inputs.jsonl` (what the writers were given), `explanations.jsonl` (full records incl. the self-check audit trail), `explanations_summary.csv` (one row per explanation), `generation_report.json` (counts and quality statistics).

In [29]:
rep = json.loads((PROC_DIR / "generation_report.json").read_text(encoding="utf-8"))
print("generator:", rep["generator"], "| cases:", rep["cases"], "| explanations:", rep["explanations_done"], "of", rep["target_main"], "(+ framed extras)")
pd.DataFrame(rep.get("by_technique", {})).T


generator: {'provider': 'ollama', 'model': 'llama3.1:8b'} | cases: 120 | explanations: 497 of 480 (+ framed extras)


,n,words_mean,words_min,words_max,share_with_number_not_in_facts
direct_llm,120.0,82.8,63.0,98.0,0.017
grounded_check_llm,120.0,78.9,47.0,115.0,0.000
grounded_llm,137.0,82.6,57.0,113.0,0.000
template,120.0,56.7,28.0,87.0,0.000


In [30]:
rep.get("check_revise")


{'share_revised': 0.35,
 'mean_revisions': 0.43,
 'share_text_changed_vs_grounded': 0.308,
 'check_failed': 0,
 'unresolved_after_revisions': 7}

In [31]:
def inspect_self_check(show: int = 5) -> None:
    """What did the auditor find in technique 4, and did the revision fix it?"""
    recs = [json.loads(l) for l in open(PROC_DIR / "explanations.jsonl", encoding="utf-8") if l.strip()]
    t4 = [r for r in recs if r["technique"] == "grounded_check_llm" and r.get("audit")]
    first, last, bad = Counter(), Counter(), []
    for r in t4:
        tr = r["audit"]["trail"]
        first.update(i["type"] for i in tr[0]["issues"])
        last.update(i["type"] for i in tr[-1]["issues"])
        if tr[-1]["issues"]:
            bad.append(r)
    print(f"{len(t4)} self-check records; {len(bad)} still have unresolved issues")
    print("issues raised in round 0 :", dict(first))
    print("issues left at the end   :", dict(last))
    for r in bad[:show]:
        print(f"\n[{r['key']}] {r['profile_source']} / {r['role']}\nTEXT: {r['text']}")
        for i in r["audit"]["trail"][-1]["issues"]:
            print(f"  - {i['type']}: {i['detail']}")


inspect_self_check(show=5)

120 self-check records; 7 still have unresolved issues
issues raised in round 0 : {'unsupported': 43, 'missing_tradeoff': 4, 'contradicted': 12}
issues left at the end   : {'unsupported': 8, 'contradicted': 1}

[S001-r2-chk] synthetic / weak_taste
TEXT: This recipe for Whole Wheat Yeast Free Herbed Pizza Dough is being recommended to you because it meets your stated dietary requirement of being vegetarian. It also fits within your time limit of about 30 minutes or less. However, it's worth noting that this recipe does not feature any of the foods you mentioned you like, such as pasta or cheese.
  - unsupported: The stated food dislikes include mushroom.
  - unsupported: The recipe is healthy.
  - contradicted: The person has a health goal.

[S004-r2-chk] synthetic / weak_taste
TEXT: This recipe is being recommended to you because it meets your stated dietary requirement of being nut-free. The ingredients and cooking time also fit within your specified limits. However, I should note tha

In [32]:
def show_case(case_key: str) -> None:
    """All four explanations for one case, e.g. show_case("H001-r1")."""
    df = pd.read_csv(PROC_DIR / "explanations_summary.csv")
    for _, r in df[df["case_key"] == case_key].iterrows():
        print(f"--- {r['technique']} ({r['framing']}, {r['words']} words) ---\n{r['text']}\n")


show_case("H001-r1")

--- template (neutral, 68 words) ---
Sweetslady's Chocolate Peanut Butter Brownie Trifle is one option we can offer you. It features desserts, peanut butter and vanilla, and your past reviews suggest you tend to like recipes with these. It takes about 60 minutes. It differs from the usual pattern of your favourites, which tend to have lower fiber. Per serving it has about 508 kcal, 8 g of protein and 272 mg of sodium.

--- direct_llm (neutral, 82 words) ---
I think you'll love Sweetslady's Chocolate Peanut Butter Brownie Trifle! Given your fondness for cookies and brownies, this recipe combines the best of both worlds in a layered dessert. The addition of peanut butter and cream cheese gives it a rich and creamy texture, while the chocolate syrup adds an extra indulgent touch. With its high calorie and sugar content, this treat is sure to satisfy your sweet tooth, and the use of powdered sugar and icing sugar aligns with your preferences.

--- grounded_llm (neutral, 101 words) ---
Base

**Next step (Step 6, case bank):** re-check the dietary labels of the recipes used, confirm the self-check technique behaves sensibly, then freeze the 480 explanations plus ~15 diagnostic cases.